STEP 1- WAKE UP

Run the box below.You should see a friendly message.

In [1]:
print("Hello! I am your smart searcher helper")
print("give me a puzzle and i will findn the shoretest path.")

Hello! I am your smart searcher helper
give me a puzzle and i will findn the shoretest path.


STEP 2

In [2]:
def astar(start, goal, get_neighbours, guess):
    # The to-do list. Each note is [f, g, place, path_so_far].
    frontier = [[guess(start), 0, start, [start]]]
    visited = []            # places we already explored
    explored_count = 0      # how many places we explored (for comparing)

    while len(frontier) > 0:
        # 1) Find the note with the SMALLEST f by looking through the list.
        best = 0
        for i in range(len(frontier)):
            if frontier[i][0] < frontier[best][0]:
                best = i
        note = frontier.pop(best)     # take that note out of the list

        f, g, place, path = note      # unpack the note

        # 2) If we already explored this place, skip it.
        if place in visited:
            continue
        visited.append(place)
        explored_count = explored_count + 1

        # 3) Did we reach the goal? Then we are done.
        if place == goal:
            return path, explored_count

        # 4) Add each neighbour to the to-do list as a new note.
        for nxt in get_neighbours(place):
            if nxt not in visited:
                new_g = g + 1                       # one more step
                new_f = new_g + guess(nxt)          # f = g + h
                frontier.append([new_f, new_g, nxt, path + [nxt]])

    return None, explored_count     # no path found

print("A* is ready. ✅")

A* is ready. ✅


STEP 3 - BUILD THE GRID

In [3]:
grid = [
    "...........",
    "....###....",
    "S.........G",
    "....###....",
    "...........",
]

def find(letter):
    for r in range(len(grid)):
        for c in range(len(grid[r])):
            if grid[r][c] == letter:
                return (r, c)

start = find("S")
goal  = find("G")
rows, cols = len(grid), len(grid[0])

print("Start S is at", start)
print("Goal  G is at", goal)
print()
for line in grid:

    print("  " + "  ".join(line))

Start S is at (2, 0)
Goal  G is at (2, 10)

  .  .  .  .  .  .  .  .  .  .  .
  .  .  .  .  #  #  #  .  .  .  .
  S  .  .  .  .  .  .  .  .  .  G
  .  .  .  .  #  #  #  .  .  .  .
  .  .  .  .  .  .  .  .  .  .  .


In [4]:
def grid_neighbours(box):
  r, c = box
  out = []
  for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
    nr, nc = r+dr, c+dc
    if 0 <= nr < rows and 0 <= nc < cols:
      if grid[nr][nc] != "#":
       out.append((nr, nc))
  return out

print("From the start",start,"the robot can step to:",grid_neighbours(start))

From the start (2, 0) the robot can step to: [(1, 0), (3, 0), (2, 1)]


In [5]:
def grid_zero(box):
  return 0

def grid_manhattan(box):
  r,c = box
  return abs(r-goal[0]) + abs(c-goal[1])

  #The formula h(n) = |x₂ - x₁| + |y₂ - y₁| represents the Manhattan Distance heuristic used in the A*


print("Manhattan guess from the start:",grid_manhattan(start))


Manhattan guess from the start: 10


In [6]:
path_zero, explored_zero = astar(start, goal, grid_neighbours, grid_zero)
path_manh, explored_manh = astar(start, goal, grid_neighbours, grid_manhattan)

print("Guess          Path length    Boxes explored")
print("-" * 46)
print("Zero (no hint) ", len(path_zero) - 1, "steps      ", explored_zero)
print("Manhattan      ", len(path_manh) - 1, "steps      ", explored_manh)
print()
print("Same shortest path. But Manhattan explores FEWER boxes. ⭐")

Guess          Path length    Boxes explored
----------------------------------------------
Zero (no hint)  10 steps       43
Manhattan       10 steps       11

Same shortest path. But Manhattan explores FEWER boxes. ⭐


In [23]:
def show_grid_path(path):
  path_set = set(path)
  for r in range(rows):
    line = ""
    for c in range(cols):
      if (r,c) == start:
        line += "S "
      elif (r,c) == goal:
        line += " G"
      elif (r,c) in path_set:
        line += " * "
      else:
        line += grid[r][c] + "  "
    print(line)

print("The robot's path :")
print()
show_grid_path(path_manh)


The robot's path :

.  .  .  .  .  .  .  .  .  .  .  
.  .  .  .  #  #  #  .  .  .  .  
S  *  *  *  *  *  *  *  *  *  G
.  .  .  .  #  #  #  .  .  .  .  
.  .  .  .  .  .  .  .  .  .  .  


a*

In [25]:
goal_state = "123456780"
start_state = "123450678"

def show_puzzle(state):
  for i in range(3):
    row=state[3*i:3*i+3]
    row = row.replace("0", "_")
    print(" " + " ".join(row))

print("start:")
show_puzzle(start_state)
print("goal:")
show_puzzle(start_state)

start:
 1 2 3
 4 5 _
 6 7 8
goal:
 1 2 3
 4 5 _
 6 7 8


In [26]:
def puzzle_neighbours(state):
    out = []
    zero = state.index("0")          # where the empty space is
    r, c = zero // 3, zero % 3
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:          # stay inside the box
            new_zero = nr * 3 + nc
            tiles = list(state)
            # swap the empty space with the tile next to it
            tiles[zero], tiles[new_zero] = tiles[new_zero], tiles[zero]
            out.append("".join(tiles))
    return out

print("From the start, we can reach these arrangements:")
for s in puzzle_neighbours(start_state):
    print("  ", s)

From the start, we can reach these arrangements:
   120453678
   123458670
   123405678


In [27]:
def puzzle_zero(state):
  return 0

def puzzle_wrong_tiles(state):
  count = 0
  for i in range(9):
    if state[i] != "0" and state[i] != goal_state[i]:
      count += 1
  return count

print("wrong tiles in the start:", puzzle_wrong_tiles(start_state))

wrong tiles in the start: 3


In [29]:
ans_zero, count_zero = astar(start_state, goal_state,
                             puzzle_neighbours, puzzle_zero)
ans_smart, count_smart = astar(start_state, goal_state,
                               puzzle_neighbours, puzzle_wrong_tiles)
print("Guess          Moves to solve    Arrangements explored")
print("-" * 56)
print("Zero (no hint) ", len(ans_zero) - 1, "steps      ", count_zero)
print("Wrong tiles    ", len(ans_smart) -1, "moves      ",count_smart)
print()
print("Same number of moves. But wrong tiles explores FAR FEWER arguments. ⭐")

Guess          Moves to solve    Arrangements explored
--------------------------------------------------------
Zero (no hint)  13 steps       2710
Wrong tiles     13 moves       190

Same number of moves. But wrong tiles explores FAR FEWER arguments. ⭐


In [32]:
print("Solving the puzzle, one slide at a time :")
print()
for step , state in enumerate(ans_smart):
  print("move", step, ":")
  show_puzzle(state)
  print()

Solving the puzzle, one slide at a time :

move 0 :
 1 2 3
 4 5 _
 6 7 8

move 1 :
 1 2 3
 4 5 8
 6 7 _

move 2 :
 1 2 3
 4 5 8
 6 _ 7

move 3 :
 1 2 3
 4 5 8
 _ 6 7

move 4 :
 1 2 3
 _ 5 8
 4 6 7

move 5 :
 1 2 3
 5 _ 8
 4 6 7

move 6 :
 1 2 3
 5 6 8
 4 _ 7

move 7 :
 1 2 3
 5 6 8
 4 7 _

move 8 :
 1 2 3
 5 6 _
 4 7 8

move 9 :
 1 2 3
 5 _ 6
 4 7 8

move 10 :
 1 2 3
 _ 5 6
 4 7 8

move 11 :
 1 2 3
 4 5 6
 _ 7 8

move 12 :
 1 2 3
 4 5 6
 7 _ 8

move 13 :
 1 2 3
 4 5 6
 7 8 _

